# Week 2 — Data Preprocessing and Baseline Model

In Week 2, the main goal is to prepare the MovieLens 20M dataset for model training.

This notebook includes:

```text
1. Data preprocessing
2. userId and movieId encoding
3. Train / validation / test split
4. Baseline recommendation models
5. Evaluation using RMSE and MAE
```

The task is not classification.  
The model predicts a numerical movie rating.

```text
Input: userId, movieId
Output: predicted rating
```

In [38]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 1. File Paths

In [39]:
RATINGS_PATH = "../data/ratings.csv"
MOVIES_PATH = "../data/movies.csv"

PROCESSED_DIR = "../data/processed"
RESULTS_DIR = "../results"
MODELS_DIR = "../models"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("ratings.csv exists:", os.path.exists(RATINGS_PATH))
print("movies.csv exists:", os.path.exists(MOVIES_PATH))

ratings.csv exists: True
movies.csv exists: True


## 2. Load Dataset

For Week 2, we only need the main dataset files:

```text
ratings.csv
movies.csv
```

The `ratings.csv` file is used for training and evaluation.  
The `movies.csv` file is used to show movie titles in recommendation examples.

In [40]:
ratings = pd.read_csv(RATINGS_PATH)
movies = pd.read_csv(MOVIES_PATH)

print("Ratings loaded:", ratings.shape)
print("Movies loaded:", movies.shape)

Ratings loaded: (20000263, 4)
Movies loaded: (27278, 3)


## 3. Select Important Columns

For this project, the most important columns are:

```text
userId
movieId
rating
timestamp
```

The model will mainly use `userId` and `movieId` as input, and `rating` as the prediction target.

From `movies.csv`, we keep:

```text
movieId
title
genres
```

These columns are useful for showing recommendation results.

In [41]:
ratings = ratings[["userId", "movieId", "rating", "timestamp"]].copy()
movies = movies[["movieId", "title", "genres"]].copy()

ratings = ratings.dropna()
movies = movies.dropna()

print("Ratings after preprocessing:", ratings.shape)
print("Movies after preprocessing:", movies.shape)

ratings.head()

Ratings after preprocessing: (20000263, 4)
Movies after preprocessing: (27278, 3)


,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


## 4. Use a Sample for Faster Experiments

MovieLens 20M is a large dataset. It contains more than 20 million ratings.

For Week 2, we can use a sample to make preprocessing and baseline training faster.  
This is acceptable for early experiments.

Later, for the final version, the sample size can be increased or the full dataset can be used.

Important:

```text
SAMPLE_SIZE = 500_000
```

means that we use 500,000 ratings for faster testing.

In [42]:
SAMPLE_SIZE = 500_000
RANDOM_STATE = 42

if SAMPLE_SIZE is not None and len(ratings) > SAMPLE_SIZE:
    ratings = ratings.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

ratings = ratings.reset_index(drop=True)

print("Sample shape:", ratings.shape)

Sample shape: (500000, 4)


## 5. Encode userId and movieId

Deep learning models cannot directly use original user IDs and movie IDs if they are large or non-continuous.

For example:

```text
Original userId: 12456
Encoded user_index: 0
```

So we create new columns:

```text
user_index
movie_index
```

These encoded values start from 0 and are suitable for embedding layers in the future deep learning model.

In [43]:
ratings["user_index"] = ratings["userId"].astype("category").cat.codes
ratings["movie_index"] = ratings["movieId"].astype("category").cat.codes

num_users = ratings["user_index"].nunique()
num_movies = ratings["movie_index"].nunique()

print("Number of encoded users:", num_users)
print("Number of encoded movies:", num_movies)

ratings[["userId", "user_index", "movieId", "movie_index", "rating"]].head()

Number of encoded users: 106761
Number of encoded movies: 13152


,userId,user_index,movieId,movie_index,rating
0,122270,94308,8360,7000,3.5
1,49018,37858,32,31,2.0
2,89527,69050,109374,12775,3.5
3,106704,82291,1060,992,3.0
4,47791,36908,1732,1586,2.0


## 6. Save Mapping Tables

Mapping tables help connect original IDs with encoded IDs.

This is useful because the model will use encoded IDs, but for interpretation we may need original `userId` and `movieId`.

We save:

```text
user_mapping.csv
movie_mapping.csv
```

In [44]:
user_mapping = ratings[["userId", "user_index"]].drop_duplicates()
movie_mapping = ratings[["movieId", "movie_index"]].drop_duplicates()

user_mapping.to_csv("../data/processed/user_mapping.csv", index=False)
movie_mapping.to_csv("../data/processed/movie_mapping.csv", index=False)

print("Mapping files saved:")
print("../data/processed/user_mapping.csv")
print("../data/processed/movie_mapping.csv")

Mapping files saved:
../data/processed/user_mapping.csv
../data/processed/movie_mapping.csv


## 7. Train / Validation / Test Split

The dataset is split into three parts:

| Part | Percentage | Purpose |
|---|---:|---|
| Train | 70% | Used to train the model |
| Validation | 15% | Used to tune and compare models |
| Test | 15% | Used for final evaluation |

Important rule:

```text
The test set should not be used for model selection.
```

This helps make the final evaluation fair.

In [45]:
train_df, temp_df = train_test_split(
    ratings,
    test_size=0.30,
    random_state=RANDOM_STATE
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE
)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)

print("\nAverage rating:")
print("Train:", train_df["rating"].mean())
print("Validation:", valid_df["rating"].mean())
print("Test:", test_df["rating"].mean())

Train shape: (350000, 6)
Validation shape: (75000, 6)
Test shape: (75000, 6)

Average rating:
Train: 3.5287542857142857
Validation: 3.5294733333333332
Test: 3.528713333333333


## 8. Save Processed Data

The processed datasets are saved so that Week 3 can use them directly for the deep learning model.

Saved files:

```text
data/processed/train.csv
data/processed/valid.csv
data/processed/test.csv
```

In [46]:
train_df.to_csv("../data/processed/train.csv", index=False)
valid_df.to_csv("../data/processed/valid.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("Processed datasets saved.")

Processed datasets saved.


## 9. Evaluation Metrics

Since this is a rating prediction task, the main metrics are:

```text
RMSE
MAE
```

### RMSE

RMSE gives more penalty to large prediction errors.

```text
Lower RMSE = better model
```

### MAE

MAE shows the average absolute difference between real and predicted ratings.

```text
Lower MAE = better model
```

In [47]:
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def evaluate_model(y_true, y_pred, model_name):
    rmse = calculate_rmse(y_true, y_pred)
    mae = calculate_mae(y_true, y_pred)

    print(model_name)
    print("=" * len(model_name))
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")

    return {
        "model": model_name,
        "RMSE": rmse,
        "MAE": mae
    }

## 10. Baseline Model 1 — Global Average Rating

The simplest baseline predicts the same rating for every user-movie pair.

It uses the average rating from the training set.

Example:

```text
If average rating in train = 3.52,
then every prediction = 3.52
```

This baseline is very simple, but it is useful as a starting point.

In [48]:
global_mean = train_df["rating"].mean()

test_global_pred = np.full(len(test_df), global_mean)

global_result = evaluate_model(
    test_df["rating"],
    test_global_pred,
    "Global Average Baseline"
)

Global Average Baseline
RMSE: 1.0522
MAE:  0.8416


## 11. Baseline Model 2 — Movie Average Rating

This baseline predicts rating based on the average rating of each movie.

For example, if a movie usually receives high ratings, the model predicts a high rating for that movie.

If a movie is not found in the training set, we use the global average rating.

In [49]:
movie_mean_ratings = train_df.groupby("movieId")["rating"].mean()

test_movie_pred = test_df["movieId"].map(movie_mean_ratings)
test_movie_pred = test_movie_pred.fillna(global_mean)

movie_result = evaluate_model(
    test_df["rating"],
    test_movie_pred,
    "Movie Average Baseline"
)

Movie Average Baseline
RMSE: 0.9580
MAE:  0.7436


## 12. Baseline Model 3 — User Average Rating

This baseline predicts rating based on the average rating behavior of each user.

Some users usually give high ratings, while others are stricter and give lower ratings.

If a user is not found in the training set, we use the global average rating.

In [50]:
user_mean_ratings = train_df.groupby("userId")["rating"].mean()

test_user_pred = test_df["userId"].map(user_mean_ratings)
test_user_pred = test_user_pred.fillna(global_mean)

user_result = evaluate_model(
    test_df["rating"],
    test_user_pred,
    "User Average Baseline"
)

User Average Baseline
RMSE: 1.0870
MAE:  0.8341


## 13. Baseline Model 4 — Combined User-Movie Average

This baseline combines two ideas:

```text
user average rating
movie average rating
```

The prediction is calculated as:

```text
predicted rating = (user average + movie average) / 2
```

This model is still simple, but usually better than using only global average.

In [51]:
test_user_part = test_df["userId"].map(user_mean_ratings).fillna(global_mean)
test_movie_part = test_df["movieId"].map(movie_mean_ratings).fillna(global_mean)

test_combined_pred = (test_user_part + test_movie_part) / 2

combined_result = evaluate_model(
    test_df["rating"],
    test_combined_pred,
    "Combined User-Movie Average Baseline"
)

Combined User-Movie Average Baseline
RMSE: 0.9482
MAE:  0.7399


## 14. Compare Baseline Models

Now we compare all baseline models using RMSE and MAE.

The best baseline is the model with the lowest RMSE and MAE.

In [52]:
baseline_results = [
    global_result,
    movie_result,
    user_result,
    combined_result
]

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,model,RMSE,MAE
0,Global Average Baseline,1.052172,0.841636
1,Movie Average Baseline,0.957986,0.743593
2,User Average Baseline,1.087004,0.834105
3,Combined User-Movie Average Baseline,0.948204,0.739882


In [53]:
baseline_results = [
    global_result,
    movie_result,
    user_result,
    combined_result
]

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,model,RMSE,MAE
0,Global Average Baseline,1.052172,0.841636
1,Movie Average Baseline,0.957986,0.743593
2,User Average Baseline,1.087004,0.834105
3,Combined User-Movie Average Baseline,0.948204,0.739882


## 15. Save Baseline Results

The baseline results are saved into the `results/` folder.

These files will be used later in the final report and model comparison.

In [54]:
baseline_results_df.to_csv(
    "../results/week2_baseline_results.csv",
    index=False
)

with open("../results/week2_baseline_results.txt", "w", encoding="utf-8") as file:
    file.write("Week 2 Baseline Results\n")
    file.write("=" * 30 + "\n\n")
    file.write(baseline_results_df.to_string(index=False))

print("Baseline results saved:")
print("../results/week2_baseline_results.csv")
print("../results/week2_baseline_results.txt")

Baseline results saved:
../results/week2_baseline_results.csv
../results/week2_baseline_results.txt


## 16. Save Baseline Objects

We save baseline objects so they can be reused later.

Saved object contains:

```text
global average rating
movie average ratings
user average ratings
```

In [55]:
baseline_objects = {
    "global_mean": global_mean,
    "movie_mean_ratings": movie_mean_ratings,
    "user_mean_ratings": user_mean_ratings
}

joblib.dump(
    baseline_objects,
    "../models/week2_baseline_objects.pkl"
)

print("Baseline objects saved:")
print("../models/week2_baseline_objects.pkl")

Baseline objects saved:
../models/week2_baseline_objects.pkl


## 17. Example Prediction

This example shows how the baseline model predicts a rating for one user-movie pair from the test set.

We compare:

```text
real rating
predicted rating
```

In [56]:
example = test_df.iloc[0]

example_user = example["userId"]
example_movie = example["movieId"]
real_rating = example["rating"]

movie_title = movies.loc[
    movies["movieId"] == example_movie,
    "title"
]

if len(movie_title) > 0:
    movie_title = movie_title.values[0]
else:
    movie_title = "Unknown movie"

predicted_rating = (
    user_mean_ratings.get(example_user, global_mean)
    + movie_mean_ratings.get(example_movie, global_mean)
) / 2

print("Example Prediction")
print("=" * 20)
print("User ID:", example_user)
print("Movie ID:", example_movie)
print("Movie title:", movie_title)
print("Real rating:", real_rating)
print("Predicted rating:", round(predicted_rating, 2))

Example Prediction
User ID: 33323.0
Movie ID: 89650.0
Movie title: Ironclad (2011)
Real rating: 2.5
Predicted rating: 3.18


## 18. Simple Recommendation Example

Here we create a simple recommendation example for one user.

Steps:

```text
1. Select one user
2. Find movies that the user has not rated
3. Predict ratings for candidate movies
4. Sort movies by predicted rating
5. Recommend top 10 movies
```

In [57]:
example_user_id = test_df["userId"].iloc[0]

rated_movies = ratings[
    ratings["userId"] == example_user_id
]["movieId"].unique()

candidate_movies = movies[
    ~movies["movieId"].isin(rated_movies)
].copy()

candidate_movies = candidate_movies.sample(
    n=min(1000, len(candidate_movies)),
    random_state=RANDOM_STATE
)

candidate_movies["user_mean"] = user_mean_ratings.get(
    example_user_id,
    global_mean
)

candidate_movies["movie_mean"] = candidate_movies["movieId"].map(
    movie_mean_ratings
).fillna(global_mean)

candidate_movies["predicted_rating"] = (
    candidate_movies["user_mean"] + candidate_movies["movie_mean"]
) / 2

recommendations = candidate_movies.sort_values(
    by="predicted_rating",
    ascending=False
).head(10)

recommendations[["movieId", "title", "genres", "predicted_rating"]]

,movieId,title,genres,predicted_rating
12282,56474,My Mother (Ma mère) (2004),Drama,3.911765
1433,1471,Boys Life 2 (1997),Drama,3.911765
9176,27033,"Kingdom II, The (Riget II) (1997)",Drama|Horror|Mystery,3.911765
22624,108401,Husk (2011),Horror|Thriller,3.911765
21498,104312,"Mortal Instruments: City of Bones, The (2013)",Action|Adventure|Drama|IMAX,3.911765
8423,25842,Topper (1937),Comedy|Fantasy|Romance,3.911765
780,793,My Life and Times With Antonin Artaud (En comp...,Drama,3.745098
3550,3641,"Woman of Paris, A (1923)",Drama,3.661765
9035,26758,All the Mornings of the World (Tous les matins...,Drama|Romance,3.661765
13721,68612,Outrage (2009),Documentary,3.661765


## 19. Save Recommendation Example

The generated recommendation example is saved to the `results/` folder.

In [58]:
recommendations[["movieId", "title", "genres", "predicted_rating"]].to_csv(
    "../results/week2_example_recommendations.csv",
    index=False
)

print("Example recommendations saved:")
print("../results/week2_example_recommendations.csv")

Example recommendations saved:
../results/week2_example_recommendations.csv


## 20. Week 2 Summary

Week 2 completed the following tasks:

```text
data preprocessing
userId and movieId encoding
train / validation / test split
baseline model implementation
RMSE and MAE evaluation
simple recommendation example
```

The processed data and baseline results are now ready for Week 3.

In [59]:
print("Week 2 completed successfully.")

print("\nProcessed files:")
print("../data/processed/train.csv")
print("../data/processed/valid.csv")
print("../data/processed/test.csv")
print("../data/processed/user_mapping.csv")
print("../data/processed/movie_mapping.csv")

print("\nResult files:")
print("../results/week2_baseline_results.csv")
print("../results/week2_baseline_results.txt")
print("../results/week2_example_recommendations.csv")

print("\nModel file:")
print("../models/week2_baseline_objects.pkl")

Week 2 completed successfully.

Processed files:
../data/processed/train.csv
../data/processed/valid.csv
../data/processed/test.csv
../data/processed/user_mapping.csv
../data/processed/movie_mapping.csv

Result files:
../results/week2_baseline_results.csv
../results/week2_baseline_results.txt
../results/week2_example_recommendations.csv

Model file:
../models/week2_baseline_objects.pkl
